# Combine + players DL export

Manual notebook for Kevin Zhang (BDB 2027). Joins Combine results to player metadata and exports DL/edge prospects.

## Prerequisites (Kevin runs this locally)

1. Place competition CSVs under `Data/` relative to repo root:
   - `Data/combine_results.csv`
   - `Data/players.csv`
2. Run **this** notebook (Cell → Run All).
3. Confirm output:
   `DerivedData/combine_players_dl.csv`

**Do not** paste `Data/` or `DerivedData/` CSV contents into agent chat or Cursor context. Both dirs are cursorignored — agents should not load or summarize the raw files.

This notebook is intentionally left unexecuted in git; Kevin runs it after `Data/` is present.

## Logic

| Step | Detail |
|---|---|
| Join key | `nfl_id` (`players` ⨝ `combine_results`) |
| Filter | `nfl_position` ∈ {`DE`, `OLB`, `DT`, `NT`} |
| Output | one row per matching `nfl_id` |

## Column mapping

| Output column | Source |
|---|---|
| `nfl_id`, `display_name`, `draft_year`, `nfl_position`, `draft_overall_pick` | `players.csv` (`draft_year` from players) |
| `combine_height`, `combine_weight`, `hand_size`, `arm_length`, `ten_yd_split`, `forty`, `vertical`, `broad_jump`, `three_cone`, `short_shuttle`, `bench_reps` | `combine_results.csv` |
| `ngs_athleticism_score` | `combine_results.ngs_athleticism_score` |
| `ngs_production_score` | `combine_results.ngs_college_production_score` (renamed) |
| `ngs_final_score` | `combine_results.ngs_final_score` |


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

# Paths relative to repo root (notebook lives in notebooks/cursor_dev/)
REPO_ROOT = Path("../..").resolve()
DATA_DIR = REPO_ROOT / "Data"
DERIVED_DIR = REPO_ROOT / "DerivedData"

PLAYERS = DATA_DIR / "players.csv"
COMBINE = DATA_DIR / "combine_results.csv"
OUT_CSV = DERIVED_DIR / "combine_players_dl.csv"

ROSTER_KEEP = ["DE", "OLB", "DT", "NT"]

# Output columns in exact order (schema rename applied below)
OUT_COLS = [
    "nfl_id",
    "display_name",
    "draft_year",
    "nfl_position",
    "draft_overall_pick",
    "combine_height",
    "combine_weight",
    "hand_size",
    "arm_length",
    "ten_yd_split",
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_production_score",
    "ngs_final_score",
]

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR exists:", DATA_DIR.is_dir())
for p in (PLAYERS, COMBINE):
    print(f"  {p.name}: {'OK' if p.is_file() else 'MISSING'} -> {p}")
print("Output:", OUT_CSV)


## Load, join, filter, export

Inner-join on `nfl_id`. Keep players-side `draft_year` when both tables have it. Rename `ngs_college_production_score` → `ngs_production_score`.


In [ ]:
assert PLAYERS.is_file(), f"Missing {PLAYERS}. Place competition CSVs under Data/."
assert COMBINE.is_file(), f"Missing {COMBINE}. Place competition CSVs under Data/."

players = pd.read_csv(PLAYERS)
combine = pd.read_csv(COMBINE)

print("players shape:", players.shape)
print("combine shape:", combine.shape)

player_cols = [
    "nfl_id",
    "display_name",
    "draft_year",
    "nfl_position",
    "draft_overall_pick",
]
combine_cols = [
    "nfl_id",
    "combine_height",
    "combine_weight",
    "hand_size",
    "arm_length",
    "ten_yd_split",
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score",
]

missing_players = [c for c in player_cols if c not in players.columns]
missing_combine = [c for c in combine_cols if c not in combine.columns]
assert not missing_players, f"players.csv missing columns: {missing_players}"
assert not missing_combine, f"combine_results.csv missing columns: {missing_combine}"

merged = players[player_cols].merge(
    combine[combine_cols],
    on="nfl_id",
    how="inner",
    validate="one_to_one",
)

before = len(merged)
dl = merged[merged["nfl_position"].isin(ROSTER_KEEP)].copy()
print(f"After join: {before} rows; after position filter {ROSTER_KEEP}: {len(dl)}")

dl = dl.rename(columns={"ngs_college_production_score": "ngs_production_score"})
dl = dl[OUT_COLS].sort_values(["draft_year", "draft_overall_pick", "nfl_id"], na_position="last")

DERIVED_DIR.mkdir(parents=True, exist_ok=True)
dl.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(dl)} rows × {len(OUT_COLS)} cols -> {OUT_CSV}")
print("Columns:", list(dl.columns))
